<a href="https://colab.research.google.com/github/omargarawani/asl-to-arabic-/blob/main/gru_seq2seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import json
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GRU, Dense, Embedding
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

In [ ]:
with open('splits.json', 'r', encoding='utf-8') as f:
    splits = json.load(f)

train_pairs = splits['train']
val_pairs   = splits['val']
test_pairs  = splits['test']

all_pairs     = train_pairs + val_pairs + test_pairs
english_words = [p[0].lower().strip() for p in all_pairs]
arabic_words  = [p[1] for p in all_pairs]

print(f"Train: {len(train_pairs)} | Val: {len(val_pairs)} | Test: {len(test_pairs)}")
print("Sample:", train_pairs[:3])

Train: 4000 | Val: 500 | Test: 500
Sample: [['kareemsami', 'كاريمسامي'], ['stephanieing', 'ستيفانياينج'], ['henryity', 'هينريتي']]


In [ ]:
PAD   = '<PAD>'
START = '<START>'
END   = '<END>'

def build_vocab(words):
    chars = sorted(set(ch for w in words for ch in w))
    vocab = [PAD, START, END] + chars
    w2i   = {c: i for i, c in enumerate(vocab)}
    i2w   = {i: c for c, i in w2i.items()}
    return vocab, w2i, i2w

eng_vocab, eng_w2i, eng_i2w = build_vocab(english_words)
ara_vocab, ara_w2i, ara_i2w = build_vocab(arabic_words)

ENG_VOCAB_SIZE = len(eng_vocab)
ARA_VOCAB_SIZE = len(ara_vocab)
ENC_MAX_LEN    = max(len(w) for w in english_words)
DEC_MAX_LEN    = max(len(w) for w in arabic_words) + 2

print(f"English vocab: {ENG_VOCAB_SIZE} | max len: {ENC_MAX_LEN}")
print(f"Arabic  vocab: {ARA_VOCAB_SIZE} | max len: {DEC_MAX_LEN}")

English vocab: 28 | max len: 15
Arabic  vocab: 23 | max len: 16


In [ ]:
def encode_eng(word):
    ids = [eng_w2i.get(ch, 0) for ch in word]
    ids += [eng_w2i[PAD]] * (ENC_MAX_LEN - len(ids))
    return ids

def encode_ara(word):
    ids = [ara_w2i[START]] + [ara_w2i.get(ch, 0) for ch in word] + [ara_w2i[END]]
    ids += [ara_w2i[PAD]] * (DEC_MAX_LEN - len(ids))
    return ids

encoder_inputs     = np.array([encode_eng(w) for w in english_words])
dec_full           = np.array([encode_ara(w) for w in arabic_words])
decoder_inputs     = dec_full[:, :-1]
decoder_targets    = dec_full[:, 1:]
decoder_targets_oh = tf.keras.utils.to_categorical(decoder_targets, num_classes=ARA_VOCAB_SIZE)

print("encoder_inputs :", encoder_inputs.shape)
print("decoder_inputs :", decoder_inputs.shape)
print("decoder_targets:", decoder_targets_oh.shape)

encoder_inputs : (5000, 15)
decoder_inputs : (5000, 15)
decoder_targets: (5000, 15, 23)


In [ ]:
EMBED_DIM  = 64
LATENT_DIM = 256

# Encoder
enc_input  = Input(shape=(ENC_MAX_LEN,), name='enc_input')
enc_embed  = Embedding(ENG_VOCAB_SIZE, EMBED_DIM, mask_zero=True, name='enc_embedding')(enc_input)
_, state_h = GRU(LATENT_DIM, return_state=True, name='enc_gru')(enc_embed)

# Decoder
dec_input  = Input(shape=(DEC_MAX_LEN - 1,), name='dec_input')
dec_embed  = Embedding(ARA_VOCAB_SIZE, EMBED_DIM, mask_zero=True, name='dec_embedding')(dec_input)
dec_gru    = GRU(LATENT_DIM, return_sequences=True, return_state=True, name='dec_gru')
dec_out, _ = dec_gru(dec_embed, initial_state=state_h)
dec_dense  = Dense(ARA_VOCAB_SIZE, activation='softmax', name='dec_output')(dec_out)

model = Model([enc_input, dec_input], dec_dense)
model.compile(optimizer=Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ enc_input           │ (None, 15)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_input           │ (None, 15)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_embedding       │ (None, 15, 64)    │      1,792 │ enc_input[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 15)        │          0 │ enc_input[0][0]   │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_embedding       │ (None, 15, 64)    │      1,472 │ dec_input[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_gru (GRU)       │ [(None, 256),     │    247,296 │ enc_embedding[0]… │
│                     │ (None, 256)]      │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_gru (GRU)       │ [(None, 15, 256), │    247,296 │ dec_embedding[0]… │
│                     │ (None, 256)]      │            │ enc_gru[0][1]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_output (Dense)  │ (None, 15, 23)    │      5,911 │ dec_gru[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 503,767 (1.92 MB)

 Trainable params: 503,767 (1.92 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
checkpoint = ModelCheckpoint('gru_seq2seq.h5', monitor='val_loss', save_best_only=True)

history = model.fit(
    [encoder_inputs, decoder_inputs],
    decoder_targets_oh,
    batch_size=64,
    epochs=50,
    validation_split=0.1,
    callbacks=[early_stop, checkpoint]
)

model.save('gru_seq2seq.h5')
print('Saved → gru_seq2seq.h5')

Epoch 1/50
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step - accuracy: 0.3867 - loss: 2.0019

71/71 ━━━━━━━━━━━━━━━━━━━━ 12s 170ms/step - accuracy: 0.4037 - loss: 1.9082 - val_accuracy: 0.4057 - val_loss: 1.8590
Epoch 2/50
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 157ms/step - accuracy: 0.4214 - loss: 1.7742

71/71 ━━━━━━━━━━━━━━━━━━━━ 12s 163ms/step - accuracy: 0.4269 - loss: 1.7479 - val_accuracy: 0.4300 - val_loss: 1.7035
Epoch 3/50
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step - accuracy: 0.4473 - loss: 1.6807

71/71 ━━━━━━━━━━━━━━━━━━━━ 12s 167ms/step - accuracy: 0.4596 - loss: 1.6519 - val_accuracy: 0.4891 - val_loss: 1.5764
Epoch 4/50
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 174ms/step - accuracy: 0.5057 - loss: 1.5408

71/71 ━━━━━━━━━━━━━━━━━━━━ 14s 190ms/step - accuracy: 0.5168 - loss: 1.5073 - val_accuracy: 0.5176 - val_loss: 1.4783
Epoch 5/50
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 192ms/step - accuracy: 0.5531 - loss: 1.4089

...

71/71 ━━━━━━━━━━━━━━━━━━━━ 11s 162ms/step - accuracy: 0.9995 - loss: 0.0076 - val_accuracy: 0.9793 - val_loss: 0.0694
Epoch 38/50
71/71 ━━━━━━━━━━━━━━━━━━━━ 11s 161ms/step - accuracy: 0.9998 - loss: 0.0060 - val_accuracy: 0.9789 - val_loss: 0.0708
Epoch 39/50
71/71 ━━━━━━━━━━━━━━━━━━━━ 19s 139ms/step - accuracy: 0.9944 - loss: 0.0235 - val_accuracy: 0.9518 - val_loss: 0.1829
Epoch 40/50
71/71 ━━━━━━━━━━━━━━━━━━━━ 11s 159ms/step - accuracy: 0.9846 - loss: 0.0525 - val_accuracy: 0.9739 - val_loss: 0.0994
Epoch 41/50
71/71 ━━━━━━━━━━━━━━━━━━━━ 20s 161ms/step - accuracy: 0.9967 - loss: 0.0178 - val_accuracy: 0.9789 - val_loss: 0.0749
Epoch 42/50
71/71 ━━━━━━━━━━━━━━━━━━━━ 21s 165ms/step - accuracy: 0.9991 - loss: 0.0091 - val_accuracy: 0.9809 - val_loss: 0.0713
Epoch 43/50
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step - accuracy: 0.9994 - loss: 0.0081

71/71 ━━━━━━━━━━━━━━━━━━━━ 20s 164ms/step - accuracy: 0.9994 - loss: 0.0071 - val_accuracy: 0.9815 - val_loss: 0.0627
Epoch 44/50
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step - accuracy: 0.9996 - loss: 0.0048

71/71 ━━━━━━━━━━━━━━━━━━━━ 11s 161ms/step - accuracy: 0.9998 - loss: 0.0043 - val_accuracy: 0.9847 - val_loss: 0.0558
Epoch 45/50
71/71 ━━━━━━━━━━━━━━━━━━━━ 21s 165ms/step - accuracy: 0.9999 - loss: 0.0038 - val_accuracy: 0.9829 - val_loss: 0.0574
Epoch 46/50
71/71 ━━━━━━━━━━━━━━━━━━━━ 20s 159ms/step - accuracy: 0.9999 - loss: 0.0030 - val_accuracy: 0.9837 - val_loss: 0.0569
Epoch 47/50
71/71 ━━━━━━━━━━━━━━━━━━━━ 11s 162ms/step - accuracy: 1.0000 - loss: 0.0028 - val_accuracy: 0.9835 - val_loss: 0.0612
Epoch 48/50
71/71 ━━━━━━━━━━━━━━━━━━━━ 11s 162ms/step - accuracy: 1.0000 - loss: 0.0025 - val_accuracy: 0.9837 - val_loss: 0.0572
Epoch 49/50
71/71 ━━━━━━━━━━━━━━━━━━━━ 12s 162ms/step - accuracy: 1.0000 - loss: 0.0023 - val_accuracy: 0.9833 - val_loss: 0.0569
Epoch 50/50
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/step - accuracy: 1.0000 - loss: 0.0021

71/71 ━━━━━━━━━━━━━━━━━━━━ 11s 162ms/step - accuracy: 1.0000 - loss: 0.0020 - val_accuracy: 0.9839 - val_loss: 0.0558


Saved → gru_seq2seq.h5


In [ ]:
# Encoder: English input → hidden state
inf_enc_model = Model(enc_input, state_h)

# Decoder: 1 char + state → next char + new state
inf_h_in   = Input(shape=(LATENT_DIM,), name='inf_h')
inf_dec_in = Input(shape=(1,),          name='inf_dec_in')

inf_embed      = model.get_layer('dec_embedding')(inf_dec_in)
inf_out, inf_h = model.get_layer('dec_gru')(inf_embed, initial_state=inf_h_in)
inf_dense_out  = model.get_layer('dec_output')(inf_out)

inf_dec_model = Model(
    [inf_dec_in, inf_h_in],
    [inf_dense_out, inf_h]
)

In [ ]:
def transliterate(english_word):
    """
    Takes an English word and returns its Arabic transliteration.
    Used by the main pipeline after the CNN predicts letters.

    Example:
        transliterate('ahmed')    → 'احمد'
        transliterate('computer') → 'كمبيوتر'
    """
    word = english_word.lower().strip()

    ids = [eng_w2i.get(ch, 0) for ch in word]
    ids += [eng_w2i[PAD]] * (ENC_MAX_LEN - len(ids))
    enc_in = np.array([ids])

    # GRU returns single state (not h and c like LSTM)
    state = inf_enc_model.predict(enc_in, verbose=0)

    target = np.array([[ara_w2i[START]]])
    result = []

    for _ in range(30):
        out, state = inf_dec_model.predict([target, state], verbose=0)
        pred_id    = np.argmax(out[0, 0, :])
        pred_char  = ara_i2w[pred_id]

        if pred_char in (END, PAD):
            break

        result.append(pred_char)
        target = np.array([[pred_id]])

    return ''.join(result)


# ── Test on test split ────────────────────────────────────────────────────────
print(f"{'English':20s} {'Predicted':20s} {'Ground Truth'}")
print('-' * 60)
for eng, ara_gt in test_pairs[:10]:
    pred = transliterate(eng)
    print(f"{eng:20s} {pred:20s} {ara_gt}")

English              Predicted            Ground Truth
------------------------------------------------------------
joeless              جويليس               جويليس
virginiable          فيرجينيابلي          فيرجينيابلي
marieed              ماريد                ماريد
teresaawi            تيريساوي             تيريساوي
emilyate             يميلياتي             يميلياتي
tylerson             تيليرسون             تيليرسون
eugenean             يوجينيان             يوجينيان
thomasson            ثوماسسون             ثوماسون
joshuaive            جوشوايفي             جوشوايفي
henryling            هينريلينج            هينريلينج
